# PyTorch 常用工具速览

> 参考索引：本 Notebook 用于环境检查和按需查询，不属于线性学习主线。请先阅读 README 中的 canonical learning path。

这份 Notebook 是一张 PyTorch 工具地图。目标不是记住所有 API，而是先知道：PyTorch 能做什么、常用工具属于哪一类、以后遇到问题该去哪里找。

> 建议：按顺序运行一遍，观察输出；暂时不理解的细节可以先跳过。

## 0. 导入与版本

PyTorch 通常简称为 `torch`。张量（Tensor）是最核心的数据结构，可以把它理解为支持 GPU 和自动求导的多维数组。

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

print("PyTorch version:", torch.__version__)

PyTorch version: 2.8.0.dev20250525


## 1. 创建张量

最常用的创建方式：

| 工具 | 用途 |
| --- | --- |
| `torch.tensor(data)` | 从 Python 列表或现有数据创建 |
| `torch.zeros(shape)` | 全 0 张量 |
| `torch.ones(shape)` | 全 1 张量 |
| `torch.full(shape, value)` | 填充指定值 |
| `torch.empty(shape)` | 只分配内存，不初始化数值 |
| `torch.arange(start, end, step)` | 等间隔整数序列，不包含终点 |
| `torch.linspace(start, end, steps)` | 在区间中取固定数量的点 |
| `torch.eye(n)` | 单位矩阵 |
| `torch.zeros_like(x)` 等 | 创建与 `x` 形状、类型、设备相同的张量 |

In [22]:
a = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
print("tensor:\n", a)
print("zeros:\n", torch.zeros(2, 3))
print("ones:\n", torch.ones(2, 3))
print("full:\n", torch.full((2, 3), 7))
print("arange:", torch.arange(0, 10, 2))
print("linspace:", torch.linspace(0, 1, 5))
print("eye:\n", torch.eye(3))
print("zeros_like:\n", torch.zeros_like(a))

tensor:
 tensor([[1., 2.],
        [3., 4.]])
zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
ones:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
full:
 tensor([[7, 7, 7],
        [7, 7, 7]])
arange: tensor([0, 2, 4, 6, 8])
linspace: tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
eye:
 tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
zeros_like:
 tensor([[0., 0.],
        [0., 0.]])


## 2. 随机数与随机分布

随机张量常用于初始化参数、生成模拟数据和打乱样本。

| 工具 | 分布或用途 |
| --- | --- |
| `torch.rand(shape)` | `[0, 1)` 均匀分布 |
| `torch.randn(shape)` | 标准正态分布，均值 0、标准差 1 |
| `torch.normal(mean, std, size)` | 自定义均值和标准差的正态分布 |
| `torch.randint(low, high, shape)` | 指定区间的随机整数 |
| `torch.randperm(n)` | `0` 到 `n-1` 的随机排列 |
| `torch.bernoulli(p)` | 按概率生成 0 或 1 |
| `torch.multinomial(weights, n)` | 按权重抽样 |
| `torch.manual_seed(seed)` | 固定随机种子，便于复现实验 |

In [4]:
torch.manual_seed(42)
print("rand:", torch.rand(4))
print("randn:", torch.randn(4))
print("normal:", torch.normal(mean=10.0, std=2.0, size=(4,)))
print("randint:", torch.randint(0, 10, (4,)))
print("randperm:", torch.randperm(6))
print("bernoulli:", torch.bernoulli(torch.tensor([0.2, 0.5, 0.8])))

rand: tensor([0.8823, 0.9150, 0.3829, 0.9593])
randn: tensor([ 0.2345,  0.2303, -1.1229, -0.1863])
normal: tensor([14.4164,  8.7240, 10.9233, 10.5347])
randint: tensor([9, 6, 3, 1])
randperm: tensor([5, 4, 3, 2, 0, 1])
bernoulli: tensor([1., 0., 1.])


### 分布对象

需要更完整的概率分布功能时，使用 `torch.distributions`。常见对象包括 `Normal`、`Uniform`、`Bernoulli`、`Categorical`。它们可以采样，也可以计算概率或对数概率。

In [5]:
normal_dist = torch.distributions.Normal(loc=0.0, scale=1.0)
samples = normal_dist.sample((5,))
print("samples:", samples)
print("log probabilities:", normal_dist.log_prob(samples))

samples: tensor([ 1.3221,  0.8172, -0.7658, -0.7506,  1.3525])
log probabilities: tensor([-1.7930, -1.2528, -1.2122, -1.2007, -1.8336])


## 3. 查看张量信息与类型转换

遇到错误时，优先查看 `shape`、`dtype` 和 `device`。

| 工具 | 用途 |
| --- | --- |
| `x.shape` / `x.size()` | 各维度大小 |
| `x.ndim` | 维度数量 |
| `x.numel()` | 元素总数 |
| `x.dtype` | 数据类型 |
| `x.device` | 所在设备 |
| `x.float()` / `x.long()` / `x.to(dtype)` | 转换数据类型 |
| `x.item()` | 单元素张量转为 Python 标量 |
| `x.tolist()` | 张量转为 Python 列表 |
| `torch.from_numpy(array)` / `x.numpy()` | 与 NumPy 互转（CPU 张量） |

In [6]:
x = torch.arange(12).reshape(3, 4)
print("shape:", x.shape, "ndim:", x.ndim, "numel:", x.numel())
print("dtype:", x.dtype, "device:", x.device)
print("as float:", x.float().dtype)
print("scalar item:", x[0, 0].item())

shape: torch.Size([3, 4]) ndim: 2 numel: 12
dtype: torch.int64 device: cpu
as float: torch.float32
scalar item: 0


## 4. 改变形状和维度

| 工具 | 用途 |
| --- | --- |
| `reshape` / `view` | 改变形状；`reshape` 通常更稳妥 |
| `flatten` | 展平多个维度 |
| `squeeze` | 删除大小为 1 的维度 |
| `unsqueeze` | 插入大小为 1 的维度 |
| `transpose` | 交换两个维度 |
| `permute` | 任意重排多个维度 |
| `expand` | 广播式扩展，不复制数据 |
| `repeat` | 重复数据，会真正复制 |

In [7]:
x = torch.arange(12).reshape(3, 4)
print("reshape:", x.reshape(2, 6).shape)
print("flatten:", x.flatten().shape)
print("unsqueeze:", x.unsqueeze(0).shape)
print("transpose:", x.transpose(0, 1).shape)
image_batch = torch.rand(8, 3, 32, 32)  # N, C, H, W
print("permute NCHW -> NHWC:", image_batch.permute(0, 2, 3, 1).shape)

reshape: torch.Size([2, 6])
flatten: torch.Size([12])
unsqueeze: torch.Size([1, 3, 4])
transpose: torch.Size([4, 3])
permute NCHW -> NHWC: torch.Size([8, 32, 32, 3])


## 5. 索引、切片、筛选与组合

张量支持类似 NumPy 的索引。常用组合工具有：

| 工具 | 用途 |
| --- | --- |
| `x[...]` | 索引和切片 |
| 布尔索引 | 按条件筛选 |
| `torch.where` | 按条件选择两个值 |
| `torch.cat` | 沿已有维度拼接 |
| `torch.stack` | 创建新维度后堆叠 |
| `torch.split` / `torch.chunk` | 拆分张量 |
| `gather` / `scatter` | 按索引收集或写入，进阶模型中常见 |

In [8]:
x = torch.arange(12).reshape(3, 4)
print("first row:", x[0])
print("last two columns:\n", x[:, -2:])
print("values > 5:", x[x > 5])
print("where:\n", torch.where(x > 5, x, torch.tensor(-1)))
print("cat shape:", torch.cat([x, x], dim=0).shape)
print("stack shape:", torch.stack([x, x], dim=0).shape)

first row: tensor([0, 1, 2, 3])
last two columns:
 tensor([[ 2,  3],
        [ 6,  7],
        [10, 11]])
values > 5: tensor([ 6,  7,  8,  9, 10, 11])
where:
 tensor([[-1, -1, -1, -1],
        [-1, -1,  6,  7],
        [ 8,  9, 10, 11]])
cat shape: torch.Size([6, 4])
stack shape: torch.Size([2, 3, 4])


## 6. 数学运算与广播

### 逐元素运算

`+`、`-`、`*`、`/`、`**` 都是逐元素运算。常见函数还有 `abs`、`sqrt`、`exp`、`log`、`sin`、`cos`、`clamp`。形状不同时，PyTorch 会尝试广播（从末尾维度对齐）。

### 聚合运算

`sum`、`mean`、`min`、`max`、`argmin`、`argmax`、`std`、`var` 会把许多数值汇总为更少的数值。通过 `dim` 指定沿哪个维度计算，`keepdim=True` 可以保留该维度。

In [9]:
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
bias = torch.tensor([10.0, 20.0, 30.0])
print("broadcast add:\n", x + bias)
print("sqrt:\n", torch.sqrt(x))
print("clamp:\n", torch.clamp(x, min=2.0, max=5.0))
print("total mean:", x.mean())
print("column sums:", x.sum(dim=0))
print("row argmax:", x.argmax(dim=1))

broadcast add:
 tensor([[11., 22., 33.],
        [14., 25., 36.]])
sqrt:
 tensor([[1.0000, 1.4142, 1.7321],
        [2.0000, 2.2361, 2.4495]])
clamp:
 tensor([[2., 2., 3.],
        [4., 5., 5.]])
total mean: tensor(3.5000)
column sums: tensor([5., 7., 9.])
row argmax: tensor([2, 2])


## 7. 线性代数

| 工具 | 用途 |
| --- | --- |
| `@` / `torch.matmul` | 矩阵乘法，支持批次维度 |
| `torch.mm` | 两个二维矩阵相乘 |
| `torch.bmm` | 批量矩阵乘法 |
| `torch.dot` | 两个一维向量点积 |
| `torch.einsum` | 用下标表达复杂张量运算 |
| `torch.linalg.norm` | 范数 |
| `torch.linalg.solve` | 解线性方程组 |
| `torch.linalg.inv` / `svd` / `eig` | 逆、奇异值分解、特征分解 |

In [10]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[2.0, 0.0], [1.0, 2.0]])
print("matrix multiply:\n", a @ b)
print("elementwise multiply:\n", a * b)
print("norm:", torch.linalg.norm(a))
print("solve Ax=b:", torch.linalg.solve(a, torch.tensor([1.0, 0.0])))

matrix multiply:
 tensor([[ 4.,  4.],
        [10.,  8.]])
elementwise multiply:
 tensor([[2., 0.],
        [3., 8.]])
norm: tensor(5.4772)
solve Ax=b: tensor([-2.0000,  1.5000])


## 8. 比较、排序与常用统计

| 工具 | 用途 |
| --- | --- |
| `torch.eq` 或 `==` | 逐元素比较 |
| `torch.isclose` / `torch.allclose` | 浮点数近似比较 |
| `torch.sort` / `torch.argsort` | 排序并返回值或索引 |
| `torch.topk` | 取最大的 k 个值和索引 |
| `torch.unique` | 去重 |
| `torch.bincount` | 统计非负整数出现次数 |
| `torch.any` / `torch.all` | 判断是否任意或全部满足条件 |
| `torch.isnan` / `torch.isinf` / `torch.isfinite` | 检查异常数值 |

In [11]:
scores = torch.tensor([0.2, 0.9, 0.4, 0.7])
print("top-2:", torch.topk(scores, k=2))
print("sorted:", torch.sort(scores))
labels = torch.tensor([0, 1, 1, 2, 2, 2])
print("counts:", torch.bincount(labels))
print("all finite:", torch.isfinite(scores).all().item())

top-2: torch.return_types.topk(
values=tensor([0.9000, 0.7000]),
indices=tensor([1, 3]))
sorted: torch.return_types.sort(
values=tensor([0.2000, 0.4000, 0.7000, 0.9000]),
indices=tensor([0, 2, 3, 1]))
counts: tensor([1, 2, 3])
all finite: True


## 9. CPU、CUDA 与 MPS 设备

`torch.device` 表示计算设备。NVIDIA GPU 使用 `cuda`，Apple 芯片 GPU 使用 `mps`，其他情况使用 `cpu`。模型和输入必须在同一设备。

常用工具：`torch.cuda.is_available()`、`torch.backends.mps.is_available()`、`x.to(device)`、`model.to(device)`、`x.cpu()`。

In [12]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

x = torch.rand(2, 3).to(device)
print("selected device:", device)
print("tensor device:", x.device)

selected device: mps
tensor device: mps:0


## 10. 自动微分（Autograd）

PyTorch 可以记录张量运算并自动计算梯度。

| 工具 | 用途 |
| --- | --- |
| `requires_grad=True` | 要求追踪该张量的运算 |
| `loss.backward()` | 从标量损失反向计算梯度 |
| `x.grad` | 查看叶子张量累积的梯度 |
| `x.detach()` | 得到与计算图分离的张量 |
| `torch.no_grad()` | 临时关闭梯度记录 |
| `torch.inference_mode()` | 推理时更彻底地关闭 autograd 开销 |
| `torch.autograd.grad` | 直接计算指定输出对输入的梯度 |

In [13]:
w = torch.tensor(2.0, requires_grad=True)
loss = (w * 3 - 10) ** 2
loss.backward()
print("loss:", loss.item(), "gradient:", w.grad.item())

with torch.inference_mode():
    prediction = w * 3
print("inference prediction:", prediction.item())

loss: 16.0 gradient: -24.0
inference prediction: 6.0


## 11. 构建神经网络：`torch.nn`

模型通常继承 `nn.Module`，或者用 `nn.Sequential` 快速组合层。

常见层：

- 全连接：`nn.Linear`
- 卷积：`nn.Conv1d`、`nn.Conv2d`、`nn.Conv3d`
- 循环网络：`nn.RNN`、`nn.GRU`、`nn.LSTM`
- 注意力：`nn.MultiheadAttention`、`nn.Transformer`
- 归一化：`nn.BatchNorm2d`、`nn.LayerNorm`
- 正则化：`nn.Dropout`
- 激活：`nn.ReLU`、`nn.GELU`、`nn.Sigmoid`、`nn.Tanh`
- 池化：`nn.MaxPool2d`、`nn.AdaptiveAvgPool2d`
- 词嵌入：`nn.Embedding`

In [14]:
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(8, 3),
)
inputs = torch.randn(5, 4)
logits = model(inputs)
print(model)
print("input shape:", inputs.shape, "output shape:", logits.shape)
print("parameter count:", sum(p.numel() for p in model.parameters()))

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.1, inplace=False)
  (3): Linear(in_features=8, out_features=3, bias=True)
)
input shape: torch.Size([5, 4]) output shape: torch.Size([5, 3])
parameter count: 67


## 12. 激活函数与函数式 API

许多操作同时存在模块形式和函数形式。例如，层可以写成 `nn.ReLU()`，临时计算可以写成 `torch.relu(x)` 或 `torch.nn.functional.relu(x)`。

常见函数：`relu`、`gelu`、`sigmoid`、`tanh`、`softmax`、`log_softmax`、`one_hot`、`normalize`、`pad`。函数式 API 通常简称 `F`：`import torch.nn.functional as F`。

In [15]:
import torch.nn.functional as F

logits = torch.tensor([[1.0, 2.0, 0.5]])
print("relu:", F.relu(logits))
print("softmax probabilities:", F.softmax(logits, dim=1))
print("one hot:", F.one_hot(torch.tensor([0, 2, 1]), num_classes=3))

relu: tensor([[1.0000, 2.0000, 0.5000]])
softmax probabilities: tensor([[0.2312, 0.6285, 0.1402]])
one hot: tensor([[1, 0, 0],
        [0, 0, 1],
        [0, 1, 0]])


## 13. 损失函数

损失函数衡量预测与目标的差距。常见选择：

| 任务 | 常用损失 |
| --- | --- |
| 回归 | `nn.MSELoss`、`nn.L1Loss`、`nn.SmoothL1Loss` |
| 多分类 | `nn.CrossEntropyLoss` |
| 二分类或多标签分类 | `nn.BCEWithLogitsLoss` |
| 类别分布比较 | `nn.KLDivLoss` |
| 相似度学习 | `nn.CosineEmbeddingLoss`、`nn.TripletMarginLoss` |

`CrossEntropyLoss` 直接接收原始 logits，不要提前做 softmax；`BCEWithLogitsLoss` 也不要提前做 sigmoid。

In [16]:
logits = torch.tensor([[2.0, 0.5, -1.0], [0.2, 1.5, 0.3]])
targets = torch.tensor([0, 1])
classification_loss = nn.CrossEntropyLoss()(logits, targets)
print("cross entropy:", classification_loss.item())

predictions = torch.tensor([2.5, 3.5])
values = torch.tensor([3.0, 3.0])
print("MSE:", nn.MSELoss()(predictions, values).item())

cross entropy: 0.347378671169281
MSE: 0.25


## 14. 优化器与学习率调度

优化器根据梯度更新模型参数。最常见的是 `torch.optim.SGD`、`Adam`、`AdamW`、`RMSprop`。

基本顺序是：

1. `optimizer.zero_grad()` 清空旧梯度。
2. 前向计算预测和损失。
3. `loss.backward()` 计算梯度。
4. `optimizer.step()` 更新参数。

学习率调度器在训练过程中调整学习率，常见的有 `StepLR`、`CosineAnnealingLR`、`ReduceLROnPlateau`、`OneCycleLR`。

In [17]:
tiny_model = nn.Linear(1, 1)
optimizer = torch.optim.Adam(tiny_model.parameters(), lr=0.01)
x_train = torch.tensor([[1.0], [2.0], [3.0]])
y_train = torch.tensor([[2.0], [4.0], [6.0]])

optimizer.zero_grad()
loss = nn.MSELoss()(tiny_model(x_train), y_train)
loss.backward()
optimizer.step()
print("one training step, loss:", loss.item())

one training step, loss: 17.774255752563477


## 15. 数据集与 DataLoader

`Dataset` 定义如何取得单个样本，`DataLoader` 负责分批、打乱和并行读取。

常用工具：

- `TensorDataset`：直接把多个张量包装成数据集。
- `DataLoader`：按 batch 迭代数据。
- `random_split`：随机划分训练集、验证集。
- `Subset`：取数据集的子集。
- `ConcatDataset`：连接多个数据集。
- `WeightedRandomSampler`：按权重采样。
- 自定义 `Dataset`：实现 `__len__` 和 `__getitem__`。

In [18]:
features = torch.randn(20, 4)
labels = torch.randint(0, 3, (20,))
dataset = TensorDataset(features, labels)
loader = DataLoader(dataset, batch_size=6, shuffle=True)

batch_features, batch_labels = next(iter(loader))
print("dataset size:", len(dataset))
print("batch shapes:", batch_features.shape, batch_labels.shape)

dataset size: 20
batch shapes: torch.Size([6, 4]) torch.Size([6])


## 16. 训练模式、评估模式和预测

`model.train()` 启用训练行为，`model.eval()` 启用评估行为。它们主要影响 Dropout 和 BatchNorm。评估时通常再搭配 `torch.inference_mode()`。

In [19]:
model.eval()
with torch.inference_mode():
    logits = model(torch.randn(2, 4))
    probabilities = torch.softmax(logits, dim=1)
    predicted_classes = logits.argmax(dim=1)

print("probabilities:\n", probabilities)
print("predicted classes:", predicted_classes)

probabilities:
 tensor([[0.4078, 0.2271, 0.3651],
        [0.2555, 0.3637, 0.3808]])
predicted classes: tensor([0, 2])


## 17. 保存与加载

常用工具是 `torch.save` 和 `torch.load`。通常保存 `model.state_dict()`，而不是直接保存整个模型对象。恢复训练时还要保存优化器状态、epoch 和指标。

In [20]:
import io

buffer = io.BytesIO()
torch.save({"model": model.state_dict()}, buffer)
buffer.seek(0)
checkpoint = torch.load(buffer, weights_only=True)
model.load_state_dict(checkpoint["model"])
print("state dict keys:", list(checkpoint["model"].keys()))

state dict keys: ['0.weight', '0.bias', '3.weight', '3.bias']


## 18. 初始化、梯度处理与调试工具

| 工具 | 用途 |
| --- | --- |
| `torch.nn.init` | Xavier、Kaiming 等参数初始化 |
| `torch.nn.utils.clip_grad_norm_` | 裁剪梯度范数，常用于 RNN/Transformer |
| `model.parameters()` | 遍历可训练参数 |
| `model.named_parameters()` | 同时查看参数名和参数 |
| `model.state_dict()` | 查看可保存的参数和缓冲区 |
| `torch.autograd.set_detect_anomaly(True)` | 定位异常反向传播，速度较慢 |
| `torch.testing.assert_close` | 测试两个张量是否足够接近 |
| `torch.profiler` | 分析 CPU/GPU 时间和内存 |

In [21]:
linear = nn.Linear(4, 2)
nn.init.xavier_uniform_(linear.weight)
for name, parameter in linear.named_parameters():
    print(name, parameter.shape, "requires_grad=", parameter.requires_grad)

torch.testing.assert_close(torch.tensor([1.0]), torch.tensor([1.0 + 1e-6]))
print("tensor comparison passed")

weight torch.Size([2, 4]) requires_grad= True
bias torch.Size([2]) requires_grad= True
tensor comparison passed


## 19. 编译、混合精度和分布式训练（先认识名字）

这些工具在模型较大或训练较慢时再深入：

- `torch.compile(model)`：编译并优化模型执行。
- `torch.amp.autocast`：使用混合精度加速计算、节省显存。
- `torch.amp.GradScaler`：CUDA 混合精度训练时缩放梯度。
- `torch.distributed`：多进程、多 GPU 分布式通信。
- `DistributedDataParallel`：常用的多 GPU 训练封装。
- `torch.export` / `torch.onnx`：导出模型供其他运行时使用。

## 20. 一次训练的完整流程

把前面的工具串起来，典型训练流程如下：

```python
model = MyModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(num_epochs):
    model.train()
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        loss = loss_fn(logits, targets)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.inference_mode():
        # 在验证集上计算指标
        pass
```

## 21. 学习顺序建议

1. 先掌握张量创建、`shape`、索引、类型和设备。
2. 再掌握形状变换、广播、聚合和矩阵乘法。
3. 理解自动微分、`nn.Module`、损失函数和优化器。
4. 学会用 `Dataset` 和 `DataLoader` 组织数据。
5. 最后进入 CNN、RNN、Attention、混合精度和模型部署。

不需要背 API。记住类别和典型名称，使用时查阅官方文档即可。

## 22. NumPy：数组计算与PyTorch互操作

NumPy 是科学计算的基础库，PyTorch 张量与 NumPy 数组可以共享内存（CPU 张量）。

| 工具 | 用途 |
| --- | --- |
| `np.array(data)` | 从列表或嵌套结构创建数组 |
| `np.zeros/ones/arange/linspace` | 创建常用数组 |
| `arr.reshape` | 改变形状 |
| `np.concatenate/stack` | 拼接数组 |
| `arr.mean/std/sum/max/argmax` | 统计聚合 |
| `torch.from_numpy(arr)` | NumPy → PyTorch（共享内存） |
| `tensor.numpy()` | PyTorch → NumPy（共享内存，CPU 张量） |

In [ ]:
import numpy as np
import torch

# 创建数组
a = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print("array:\n", a)
print("zeros:", np.zeros((2, 3)))
print("arange:", np.arange(0, 10, 2))
print("linspace:", np.linspace(0, 1, 5))

# 统计
print("\nmean:", a.mean(), "std:", a.std(), "sum:", a.sum())
print("axis-0 mean:", a.mean(axis=0))
print("argmax:", a.argmax())

# reshape / concatenate
b = a.reshape(3, 2)
print("\nreshaped:", b.shape)
c = np.concatenate([a, a], axis=0)
print("concatenated:", c.shape)

# PyTorch 互操作（共享内存）
arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(arr)   # 共享内存
arr[0] = 99.0               # 修改 NumPy 数组
print("\nshared memory demo - tensor:", t)  # 张量也随之改变

t2 = torch.tensor([4.0, 5.0, 6.0])
arr2 = t2.numpy()           # CPU 张量 → NumPy，同样共享内存
print("tensor to numpy:", arr2)

## 23. Pandas：表格数据处理

Pandas 是处理结构化数据（CSV、Excel、数据库）的首选工具，训练前的数据清洗和特征工程通常在这里完成。

| 工具 | 用途 |
| --- | --- |
| `pd.DataFrame(data)` | 从字典或数组创建表格 |
| `pd.read_csv(path)` | 读取 CSV 文件 |
| `df.head/info/describe` | 快速查看数据 |
| `df.dropna/fillna` | 处理缺失值 |
| `df.drop_duplicates` | 去重 |
| `df[col].apply/map` | 逐元素变换 |
| `df.groupby` | 分组聚合 |
| `df.values` / `torch.tensor(df.values)` | 转为 NumPy / PyTorch 张量 |

In [ ]:
import pandas as pd
import torch

# 创建 DataFrame（模拟真实数据集）
data = {
    "age":    [25, 30, None, 22, 35, 28, 30],
    "income": [50000, 60000, 55000, None, 80000, 62000, 60000],
    "edu":    ["bachelor", "master", "bachelor", "bachelor", "phd", "master", "master"],
    "label":  [0, 1, 0, 0, 1, 1, 1],
}
df = pd.DataFrame(data)
print("head:\n", df.head(3))
print("\ninfo:"); df.info()
print("\ndescribe:\n", df.describe())

# 数据清洗
df = df.dropna()                         # 删除含缺失值的行
df = df.drop_duplicates()                # 去重
df["age"] = df["age"].astype(float)

# 特征工程
edu_map = {"bachelor": 0, "master": 1, "phd": 2}
df["edu_code"] = df["edu"].map(edu_map)
df["income_k"] = df["income"].apply(lambda x: x / 1000)  # 单位换算
print("\n处理后:\n", df[["age", "income_k", "edu_code", "label"]])

# groupby 聚合
print("\n按 edu 分组均值:\n", df.groupby("edu")["income"].mean())

# 转为 PyTorch 张量
features = df[["age", "income_k", "edu_code"]].values
labels   = df["label"].values
X = torch.tensor(features, dtype=torch.float32)
y = torch.tensor(labels,   dtype=torch.long)
print("\nfeature tensor:", X.shape, "label tensor:", y.shape)

## 24. Matplotlib：可视化

Matplotlib 是最常用的绘图库，训练中用来观察损失曲线、精度变化、样本图像等。

| 工具 | 用途 |
| --- | --- |
| `plt.plot(x, y)` | 绘制折线图（损失、精度曲线） |
| `plt.scatter(x, y)` | 散点图 |
| `plt.bar(x, height)` | 柱状图 |
| `plt.imshow(image)` | 显示图像或特征图 |
| `plt.subplot(rows, cols, index)` | 多子图布局 |
| `plt.xlabel/ylabel/title/legend` | 标注 |
| `plt.savefig(path)` | 保存图片 |

In [ ]:
import matplotlib
matplotlib.use("Agg")  # 非交互式后端，Notebook 中可去掉这行
import matplotlib.pyplot as plt
import numpy as np

# 模拟训练指标
epochs = list(range(1, 21))
train_loss = [1.0 / (0.3 * e + 1) + np.random.uniform(0, 0.05) for e in epochs]
val_loss   = [1.0 / (0.25 * e + 1) + np.random.uniform(0, 0.08) for e in epochs]
train_acc  = [1 - tl * 0.9 for tl in train_loss]
val_acc    = [1 - vl * 0.9 for vl in val_loss]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 损失曲线
axes[0].plot(epochs, train_loss, label="Train Loss", color="royalblue")
axes[0].plot(epochs, val_loss,   label="Val Loss",   color="tomato", linestyle="--")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss"); axes[0].legend()

# 精度曲线
axes[1].plot(epochs, train_acc, label="Train Acc", color="seagreen")
axes[1].plot(epochs, val_acc,   label="Val Acc",   color="orange", linestyle="--")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training & Validation Accuracy"); axes[1].legend()

plt.tight_layout()
plt.savefig("/tmp/training_curves.png", dpi=80)
plt.show()
print("图表已保存至 /tmp/training_curves.png")

## 25. Seaborn：统计可视化

Seaborn 基于 Matplotlib，提供更美观的统计图表，适合探索数据分布和特征相关性。

| 工具 | 用途 |
| --- | --- |
| `sns.histplot(data)` | 直方图，观察分布形状 |
| `sns.kdeplot(data)` | 核密度估计曲线 |
| `sns.heatmap(matrix)` | 热力图（相关矩阵、混淆矩阵） |
| `sns.pairplot(df)` | 多变量两两散点图 |
| `sns.boxplot/violinplot` | 箱线图、提琴图 |

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 模拟特征数据
np.random.seed(0)
df = pd.DataFrame({
    "feature_1": np.random.randn(200),
    "feature_2": np.random.randn(200) * 0.5 + 1,
    "feature_3": np.random.randn(200) * 2 - 1,
})
df["feature_2"] += df["feature_1"] * 0.7  # 制造相关性

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 分布图
sns.histplot(df["feature_1"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Feature 1 Distribution")

# 相关性热力图
corr = df.corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Feature Correlation")

plt.tight_layout()
plt.show()

## 26. scikit-learn：预处理与评估

scikit-learn 提供完整的传统 ML 工具链，在 PyTorch 项目中常用于数据预处理和评估指标计算。

| 工具 | 用途 |
| --- | --- |
| `train_test_split` | 划分训练集和测试集 |
| `StandardScaler` | 标准化特征（均值0，标准差1） |
| `MinMaxScaler` | 归一化到 [0, 1] |
| `LabelEncoder` | 类别标签编码为整数 |
| `accuracy_score` | 分类精度 |
| `classification_report` | 精确率、召回率、F1 |
| `confusion_matrix` | 混淆矩阵 |
| `cross_val_score` | 交叉验证评分 |

In [ ]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 模拟数据集
np.random.seed(42)
X = np.random.randn(200, 4)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

# 划分训练/测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")

# 标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # 在训练集上 fit
X_test_scaled  = scaler.transform(X_test)        # 测试集只用 transform
print(f"训练集均值: {X_train_scaled.mean(axis=0).round(3)}, 标准差: {X_train_scaled.std(axis=0).round(3)}")

# 转为 PyTorch 张量
X_tr = torch.tensor(X_train_scaled, dtype=torch.float32)
X_te = torch.tensor(X_test_scaled,  dtype=torch.float32)

# 用简单阈值模拟预测（演示评估API）
y_pred = (X_test_scaled[:, 0] + X_test_scaled[:, 1] > 0).astype(int)
print("\naccuracy:", accuracy_score(y_test, y_pred))
print("\nclassification report:\n", classification_report(y_test, y_pred))
print("confusion matrix:\n", confusion_matrix(y_test, y_pred))

## 27. tqdm：进度条

tqdm 为循环添加进度条，在训练循环中直观显示进度和指标。

| 工具 | 用途 |
| --- | --- |
| `tqdm(iterable)` | 给任意可迭代对象加进度条 |
| `trange(n)` | 等价于 `tqdm(range(n))` |
| `嵌套循环` | epoch + batch 双进度条 |
| `.set_description(text)` | 设置进度条前缀描述 |
| `.set_postfix(dict)` | 在进度条末尾显示实时指标 |
| `tqdm.notebook.tqdm` | Jupyter Notebook 专用版本 |

In [ ]:
from tqdm import tqdm
import time

# 基本用法
print("基本进度条:")
for i in tqdm(range(50), desc="Processing"):
    time.sleep(0.01)

# 嵌套进度条（模拟训练循环）
print("\n训练循环示例:")
num_epochs = 3
batches_per_epoch = 20

for epoch in tqdm(range(num_epochs), desc="Epochs"):
    epoch_loss = 0
    pbar = tqdm(range(batches_per_epoch), desc=f"Epoch {epoch+1}", leave=False)
    for batch in pbar:
        # 模拟训练步骤
        loss = 1.0 / (batch + epoch * 10 + 1)
        epoch_loss += loss
        # 实时显示指标
        pbar.set_postfix({"loss": f"{loss:.4f}", "avg_loss": f"{epoch_loss/(batch+1):.4f}"})
        time.sleep(0.02)

print("\n训练完成!")

## 28. torchvision：计算机视觉

torchvision 是 PyTorch 官方计算机视觉库，提供数据集、变换和预训练模型。

| 工具 | 用途 |
| --- | --- |
| `transforms.Compose` | 串联多个变换 |
| `transforms.ToTensor` | PIL/NumPy 图像 → 张量 |
| `transforms.Normalize` | 标准化（通道级均值/标准差） |
| `transforms.Resize/RandomCrop` | 尺寸调整、随机裁剪 |
| `transforms.RandomHorizontalFlip` | 随机水平翻转（数据增强） |
| `datasets.MNIST/CIFAR10/ImageFolder` | 常用数据集 |
| `models.resnet18/vgg16` | 预训练模型 |

In [ ]:
import torch, torch.nn as nn
from torchvision import transforms, models

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
print("训练变换:", train_tf)

model = models.resnet18(weights="IMAGENET1K_V1")
for p in model.parameters(): p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 10)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"ResNet-18 迁移学习: 可训练 {trainable:,} / 总 {total:,}")
print("\n数据集 API (需 download=True):")
print("  CIFAR10: 60000张32×32, 10类 | MNIST: 70000张28×28手写数字")
print("  ImageFolder: root/class/img.jpg → 自动生成标签")

## 29. torchmetrics：指标计算

torchmetrics 与 PyTorch 训练循环深度集成，支持多 GPU 自动累积。

| 工具 | 用途 |
| --- | --- |
| `MulticlassAccuracy(num_classes)` | 分类精度 |
| `MulticlassF1Score(num_classes)` | F1 调和均值 |
| `MulticlassConfusionMatrix` | 混淆矩阵 |
| `.update(preds, targets)` | 逐批次更新 |
| `.compute()` | 汇总全部批次 |
| `.reset()` | 清空，准备下一 epoch |

In [ ]:
import torch
try:
    from torchmetrics.classification import MulticlassAccuracy, MulticlassF1Score, MulticlassConfusionMatrix
    n = 3
    acc = MulticlassAccuracy(num_classes=n, average="macro")
    f1  = MulticlassF1Score(num_classes=n, average="macro")
    cm  = MulticlassConfusionMatrix(num_classes=n)
    torch.manual_seed(0)
    for _ in range(3):
        logits  = torch.randn(8, n)
        targets = torch.randint(0, n, (8,))
        preds   = logits.argmax(1)
        acc.update(preds, targets); f1.update(preds, targets); cm.update(preds, targets)
    print("Accuracy:", acc.compute().item())
    print("F1 Score:", f1.compute().item())
    print("Confusion Matrix:\n", cm.compute())
    acc.reset(); f1.reset(); cm.reset()
except ImportError:
    print("安装: pip install torchmetrics")

## 30. torchaudio：音频处理（简介）

| 工具 | 用途 |
| --- | --- |
| `torchaudio.load(path)` | 加载音频 → (waveform, sample_rate) |
| `transforms.Resample(orig, new)` | 重采样 |
| `transforms.Spectrogram()` | 语谱图 |
| `transforms.MelSpectrogram()` | 梅尔语谱图（符合人耳感知） |
| `transforms.MFCC()` | 梅尔倒谱系数（语音识别特征） |

安装：`pip install torchaudio`

In [ ]:
import torch
try:
    import torchaudio.transforms as AT
    waveform = torch.randn(1, 16000)  # 1s 16kHz 单声道
    print("waveform:", waveform.shape)
    print("after Resample 8kHz:", AT.Resample(16000, 8000)(waveform).shape)
    print("MelSpectrogram:", AT.MelSpectrogram(sample_rate=16000, n_mels=80)(waveform).shape)
    print("MFCC:", AT.MFCC(sample_rate=16000, n_mfcc=13)(waveform).shape)
except ImportError:
    print("安装: pip install torchaudio")
    print("核心: torchaudio.load(path) → (waveform, sr)")
    print("变换: Resample, Spectrogram, MelSpectrogram, MFCC, AmplitudeToDB")

## 31. torchtext：文本处理（简介）

| 工具 | 用途 |
| --- | --- |
| `build_vocab_from_iterator(iter)` | 从文本迭代器构建词典 |
| `vocab.lookup_indices(tokens)` | token → id |
| `get_tokenizer("basic_english")` | 基础英文分词器 |

**注意**：新项目建议优先使用 Hugging Face `tokenizers`（Section 34）。

In [ ]:
try:
    from torchtext.vocab import build_vocab_from_iterator
    from torchtext.data.utils import get_tokenizer
    tokenizer = get_tokenizer("basic_english")
    corpus = ["the cat sat on the mat", "the dog ran in the park", "deep learning is powerful"]
    vocab = build_vocab_from_iterator(
        (tokenizer(t) for t in corpus),
        specials=["<unk>","<pad>","<bos>","<eos>"]
    )
    vocab.set_default_index(vocab["<unk>"])
    tokens = tokenizer("the cat sat here")
    print("tokens:", tokens)
    print("indices:", vocab.lookup_indices(tokens))
    print("vocab size:", len(vocab))
except ImportError:
    print("安装: pip install torchtext")
    print("新项目推荐: Hugging Face tokenizers (Section 34)")

## 32. Hugging Face Transformers

数千个预训练模型，NLP / 多模态任务的事实标准。

| 工具 | 用途 |
| --- | --- |
| `AutoTokenizer.from_pretrained(name)` | 加载分词器 |
| `AutoModel.from_pretrained(name)` | 加载编码器模型 |
| `AutoModelForSequenceClassification` | 文本分类 |
| `AutoModelForCausalLM` | 因果语言模型（GPT 风格） |
| `pipeline(task)` | 一行完成常见任务 |
| `Trainer / TrainingArguments` | 封装训练循环，支持分布式 |

安装：`pip install transformers`

In [ ]:
try:
    from transformers import AutoTokenizer, AutoModel
    import torch

    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    enc = tokenizer(["Hello world!", "Deep learning is fun."],
                    padding=True, truncation=True, max_length=16, return_tensors="pt")
    print("input_ids:", enc["input_ids"].shape)

    model = AutoModel.from_pretrained("bert-base-uncased")
    with torch.inference_mode():
        out = model(**enc)
    print("last_hidden_state:", out.last_hidden_state.shape)
    print("CLS embedding:", out.last_hidden_state[:, 0, :].shape)

except Exception as e:
    print(f"需要安装 transformers 及网络: {e}")
    print()
    print("核心用法:")
    print("  tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')")
    print("  model = AutoModel.from_pretrained('bert-base-uncased')")
    print("  enc = tokenizer(text, return_tensors='pt')")
    print("  out = model(**enc)  → out.last_hidden_state: (B, seq, hidden)")
    print()
    print("pipeline 快捷方式:")
    print("  pipe = pipeline('sentiment-analysis')")
    print("  pipe('I love PyTorch!')  → [{'label': 'POSITIVE', 'score': 0.99}]")

## 33. Hugging Face Datasets

统一的数据集 API，支持本地数据和 Hub 数千个公开数据集。

| 工具 | 用途 |
| --- | --- |
| `load_dataset(name)` | 从 Hub 加载 |
| `Dataset.from_dict(d)` | 从字典创建本地数据集 |
| `.map(fn, batched=True)` | 批量预处理（并行） |
| `.filter(fn)` | 按条件过滤 |
| `.train_test_split(test_size)` | 切分训练/测试 |
| `streaming=True` | 流式加载，不下载到磁盘 |

安装：`pip install datasets`

In [ ]:
try:
    from datasets import Dataset
    import torch
    from torch.utils.data import DataLoader

    ds = Dataset.from_dict({
        "text":  ["I love NLP", "Deep learning rocks", "Transformers are great", "PyTorch is fast"],
        "label": [1, 1, 1, 1],
    })
    print("Dataset:", ds)
    print("第一条:", ds[0])

    # map 预处理
    ds = ds.map(lambda x: {"length": len(x["text"].split())})
    print("\n添加 length 列:", ds[:2])

    # filter
    long_ds = ds.filter(lambda x: x["length"] > 2)
    print(f"length > 2 的样本数: {len(long_ds)}")

    # split
    splits = ds.train_test_split(test_size=0.25, seed=42)
    print("splits:", splits)

    # DataLoader 集成
    ds.set_format("torch", columns=["label"])
    batch = next(iter(DataLoader(ds, batch_size=2)))
    print("DataLoader labels:", batch["label"])
except Exception as e:
    print(f"安装: pip install datasets  ({e})")

## 34. tokenizers：快速分词器

Hugging Face tokenizers 用 Rust 实现，支持从头训练自定义分词器。

| 工具 | 用途 |
| --- | --- |
| `Tokenizer(models.BPE())` | 创建 BPE 分词器 |
| `BpeTrainer(vocab_size, special_tokens)` | BPE 训练配置 |
| `tokenizer.train(files, trainer)` | 在语料上训练 |
| `tokenizer.encode(text).tokens` | 分词结果 |
| `tokenizer.encode(text).ids` | token id 列表 |
| `tokenizer.save(path)` | 保存分词器 |

安装：`pip install tokenizers`

In [ ]:
try:
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers
    import tempfile, os

    corpus = [
        "the quick brown fox jumps over the lazy dog",
        "machine learning is a subset of artificial intelligence",
        "deep learning uses neural networks with many layers",
        "pytorch is a popular deep learning framework",
    ]
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as f:
        f.write("\n".join(corpus)); tmp = f.name

    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(vocab_size=100, special_tokens=["[UNK]","[PAD]","[BOS]","[EOS]"])
    tokenizer.train([tmp], trainer)
    os.unlink(tmp)

    enc = tokenizer.encode("deep learning transforms everything")
    print("tokens:", enc.tokens)
    print("ids:   ", enc.ids)
    print("vocab_size:", tokenizer.get_vocab_size())
except ImportError:
    print("安装: pip install tokenizers")
    print("from tokenizers import Tokenizer, models, trainers")

## 35. sentencepiece 与 tiktoken（简介）

两个最常见的大语言模型分词工具。

### sentencepiece
Google 开发，LLaMA、T5、Gemma 等模型使用。

| 工具 | 用途 |
| --- | --- |
| `SentencePieceTrainer.train(...)` | 训练 BPE/Unigram 分词器 |
| `sp.encode(text, out_type=str)` | 文本 → tokens |
| `sp.encode(text, out_type=int)` | 文本 → ids |
| `sp.decode(ids)` | ids → 文本 |

### tiktoken
OpenAI 开发，用于 GPT-3.5/4/o1 系列。

| 工具 | 用途 |
| --- | --- |
| `tiktoken.encoding_for_model(name)` | 按模型名加载 |
| `enc.encode(text)` | 文本 → token ids |
| `enc.decode(ids)` | ids → 文本 |

安装：`pip install sentencepiece tiktoken`

In [ ]:
import tempfile, os

# sentencepiece
try:
    import sentencepiece as spm
    text = "pytorch is great for deep learning. " * 100
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as f:
        f.write(text); tmp = f.name
    with tempfile.TemporaryDirectory() as d:
        prefix = os.path.join(d, "sp")
        spm.SentencePieceTrainer.train(
            input=tmp, model_prefix=prefix,
            vocab_size=100, model_type="bpe", pad_id=0, unk_id=1,
        )
        sp = spm.SentencePieceProcessor(); sp.load(prefix + ".model")
        sample = "deep learning with pytorch"
        print("sentencepiece:", sp.encode(sample, out_type=str))
        ids = sp.encode(sample, out_type=int)
        print("ids:", ids, "→ decoded:", sp.decode(ids))
    os.unlink(tmp)
except ImportError:
    print("sentencepiece 未安装: pip install sentencepiece")

# tiktoken
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 使用
    ids = enc.encode("Hello, I am a large language model.")
    print("\ntiktoken ids:", ids)
    print("n_tokens:", len(ids), "| decoded:", enc.decode(ids))
except ImportError:
    print("\ntiktoken 未安装: pip install tiktoken")
    print("enc = tiktoken.encoding_for_model('gpt-4')")

## 36. PEFT 与 LoRA：参数高效微调

只训练少量参数，达到接近全参微调的效果。

| 工具 | 用途 |
| --- | --- |
| `LoraConfig(r, lora_alpha, target_modules)` | LoRA 超参配置 |
| `get_peft_model(model, config)` | 注入 LoRA 适配器 |
| `model.print_trainable_parameters()` | 查看可训练参数占比 |
| `TaskType.CAUSAL_LM / SEQ_CLS` | 任务类型 |
| `PeftModel.from_pretrained(base, path)` | 加载已保存的 LoRA 权重 |

**LoRA 原理**：将权重更新 ΔW 分解为两个低秩矩阵 A×B（rank << 原始维度），只训练 A 和 B，大幅减少可训练参数量。

**QLoRA**：在 4-bit 量化模型上应用 LoRA，进一步降低显存需求。

安装：`pip install peft`

In [ ]:
import torch, torch.nn as nn
try:
    from peft import LoraConfig, get_peft_model

    class TinyLM(nn.Module):
        def __init__(self):
            super().__init__()
            self.embed = nn.Embedding(1000, 128)
            self.layers = nn.ModuleList([nn.Linear(128, 128) for _ in range(4)])
            self.lm_head = nn.Linear(128, 1000)
        def forward(self, x):
            h = self.embed(x)
            for l in self.layers: h = torch.relu(l(h))
            return self.lm_head(h)

    base = TinyLM()
    total = sum(p.numel() for p in base.parameters())
    print(f"原始参数量: {total:,}")

    config = LoraConfig(
        r=4, lora_alpha=16,
        target_modules=["layers.0", "layers.1", "lm_head"],
        lora_dropout=0.05, bias="none",
    )
    model = get_peft_model(base, config)
    model.print_trainable_parameters()

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"可训练: {trainable:,} / {total:,} = {trainable/total*100:.2f}%")
except ImportError:
    print("安装: pip install peft")
    print("from peft import LoraConfig, get_peft_model")
    print("config = LoraConfig(r=8, lora_alpha=32, target_modules=['q_proj','v_proj'])")
    print("model = get_peft_model(hf_model, config)")

## 37. TensorBoard

PyTorch 原生支持 TensorBoard，用于可视化训练过程中的指标、权重和网络结构。

| 工具 | 用途 |
| --- | --- |
| `SummaryWriter(log_dir)` | 创建日志写入器 |
| `.add_scalar(tag, value, step)` | 记录标量（loss、accuracy） |
| `.add_scalars(tag, dict, step)` | 同时记录多个标量 |
| `.add_image(tag, img_tensor, step)` | 记录图像 |
| `.add_histogram(tag, tensor, step)` | 记录权重分布 |
| `.add_graph(model, input)` | 可视化网络结构 |

启动：`tensorboard --logdir=runs`，访问 `http://localhost:6006`

安装：`pip install tensorboard`

In [ ]:
import torch, torch.nn as nn, shutil, os
try:
    from torch.utils.tensorboard import SummaryWriter

    log_dir = "/tmp/tb_demo"
    if os.path.exists(log_dir): shutil.rmtree(log_dir)
    writer = SummaryWriter(log_dir=log_dir)

    model = nn.Linear(4, 2)
    opt   = torch.optim.Adam(model.parameters(), lr=0.01)
    torch.manual_seed(42)

    for step in range(50):
        x, y = torch.randn(16, 4), torch.randint(0, 2, (16,))
        logits = model(x)
        loss = nn.CrossEntropyLoss()(logits, y)
        acc  = (logits.argmax(1) == y).float().mean()
        writer.add_scalar("Loss/train", loss.item(), step)
        writer.add_scalar("Accuracy/train", acc.item(), step)
        opt.zero_grad(); loss.backward(); opt.step()

    for name, p in model.named_parameters():
        writer.add_histogram(f"params/{name}", p.data, 50)
    writer.add_graph(model, torch.randn(1, 4))
    writer.close()
    print(f"日志已写入 {log_dir}")
    print("启动: tensorboard --logdir=/tmp/tb_demo")
except ImportError:
    print("安装: pip install tensorboard")

## 38. Weights & Biases（wandb）

云端实验管理平台，支持指标追踪、团队协作和超参搜索。

| 工具 | 用途 |
| --- | --- |
| `wandb.init(project, config)` | 初始化实验 |
| `wandb.log({"loss": val, ...})` | 记录指标 |
| `wandb.watch(model)` | 追踪梯度和权重 |
| `wandb.save(path)` | 保存文件到云端 |
| `wandb.finish()` | 结束实验 |
| `wandb.sweep(config)` | 定义超参搜索空间 |

使用前需在 `wandb.ai` 注册并运行 `wandb login`。

安装：`pip install wandb`

In [ ]:
print("""wandb 完整训练流程:

import wandb

wandb.init(project="my-model", config={
    "learning_rate": 1e-3, "batch_size": 32, "epochs": 10,
})

wandb.watch(model, log="all", log_freq=10)

for epoch in range(config.epochs):
    for batch in train_loader:
        loss = train_step(batch)
        wandb.log({"train/loss": loss, "epoch": epoch})

    val_acc = evaluate(val_loader)
    wandb.log({"val/acc": val_acc})

torch.save(model.state_dict(), "best.pt")
wandb.save("best.pt")
wandb.finish()

超参搜索:
sweep_config = {
    "method": "bayes",
    "metric": {"name": "val/acc", "goal": "maximize"},
    "parameters": {
        "lr": {"min": 1e-5, "max": 1e-2},
        "batch_size": {"values": [16, 32, 64]},
    },
}
sweep_id = wandb.sweep(sweep_config, project="sweep-demo")
wandb.agent(sweep_id, function=train, count=20)
""")

## 39. PyTorch Lightning

Lightning 将训练逻辑封装进 `LightningModule`，由 `Trainer` 处理设备、分布式、日志、checkpoint 等工程细节。

| 工具 | 用途 |
| --- | --- |
| `LightningModule` | 模型+训练步骤的基类 |
| `training_step(batch, idx)` | 定义单步训练逻辑 |
| `validation_step(batch, idx)` | 定义验证步骤 |
| `configure_optimizers()` | 返回优化器和调度器 |
| `Trainer(max_epochs, ...)` | 自动处理训练循环 |
| `ModelCheckpoint` | 保存最优 checkpoint |
| `EarlyStopping` | 验证指标不改善时提前停止 |

安装：`pip install lightning`

In [ ]:
import torch, torch.nn as nn
try:
    import lightning as L
    from torch.utils.data import DataLoader, TensorDataset

    class SimpleClassifier(L.LightningModule):
        def __init__(self, lr=1e-3):
            super().__init__()
            self.save_hyperparameters()
            self.net = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3))
            self.loss_fn = nn.CrossEntropyLoss()

        def forward(self, x): return self.net(x)

        def training_step(self, batch, _):
            x, y = batch
            loss = self.loss_fn(self(x), y)
            acc  = (self(x).argmax(1) == y).float().mean()
            self.log("train_loss", loss, prog_bar=True)
            self.log("train_acc",  acc,  prog_bar=True)
            return loss

        def validation_step(self, batch, _):
            x, y = batch
            self.log("val_loss", self.loss_fn(self(x), y), prog_bar=True)

        def configure_optimizers(self):
            return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

    X = torch.randn(200, 4); y = torch.randint(0, 3, (200,))
    ds = TensorDataset(X, y)
    train_dl = DataLoader(ds[:160], batch_size=16)
    val_dl   = DataLoader(ds[160:], batch_size=16)

    model = SimpleClassifier()
    trainer = L.Trainer(max_epochs=3, enable_progress_bar=True, log_every_n_steps=5)
    trainer.fit(model, train_dl, val_dl)
    print("\n超参:", model.hparams)
except ImportError:
    print("安装: pip install lightning")
    print("继承 L.LightningModule，实现 training_step + configure_optimizers")

## 40. Accelerate：分布式训练简化

Hugging Face Accelerate 只需3行改动，即可让现有 PyTorch 代码支持多 GPU、多机和混合精度。

| 工具 | 用途 |
| --- | --- |
| `Accelerator()` | 自动检测配置训练环境 |
| `accelerator.prepare(model, opt, loader)` | 统一处理设备和分布式 |
| `accelerator.backward(loss)` | 替代 `loss.backward()` |
| `accelerator.gather(tensor)` | 跨 GPU 汇总 |
| `accelerate launch script.py` | 命令行启动分布式训练 |

安装：`pip install accelerate`

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
try:
    from accelerate import Accelerator

    accelerator = Accelerator(mixed_precision="no")  # 改动1

    model = nn.Linear(4, 2)
    opt   = torch.optim.AdamW(model.parameters(), lr=1e-3)
    X, y  = torch.randn(100, 4), torch.randint(0, 2, (100,))
    loader = DataLoader(TensorDataset(X, y), batch_size=16)

    model, opt, loader = accelerator.prepare(model, opt, loader)  # 改动2

    model.train()
    for epoch in range(2):
        for x_batch, y_batch in loader:
            opt.zero_grad()
            loss = nn.CrossEntropyLoss()(model(x_batch), y_batch)
            accelerator.backward(loss)   # 改动3
            opt.step()
        print(f"epoch {epoch+1} loss: {loss.item():.4f}")

    print("device:", accelerator.device)
    print("num_processes:", accelerator.num_processes)
except ImportError:
    print("安装: pip install accelerate")
    print("多 GPU 启动: accelerate launch --num_processes=4 train.py")

## 41. DeepSpeed（简介）

Microsoft 开发的大规模模型训练框架，核心是 ZeRO（零冗余优化器），通过分片将每张 GPU 的显存占用降低数倍到十数倍。

| 技术 | 显存优化效果 |
| --- | --- |
| **ZeRO Stage 1** | 分片优化器状态，减少约 4× |
| **ZeRO Stage 2** | 分片优化器状态 + 梯度，减少约 8× |
| **ZeRO Stage 3** | 分片所有内容（含模型参数），理论无限扩展 |
| **ZeRO-Offload** | 优化器状态卸载到 CPU |
| **ZeRO-Infinity** | 参数/梯度卸载到 NVMe |

**与 Transformers Trainer 集成**：在 `TrainingArguments` 中指定 `deepspeed="ds_config.json"` 即可启用。

**环境要求**：CUDA + NCCL，通常在多 GPU 服务器使用。

安装：`pip install deepspeed`（需要 CUDA 编译器）

## 42. Flash Attention（简介）

IO 感知的精确注意力算法，通过分块计算避免将完整注意力矩阵写入显存，不改变数学结果。

| 特性 | 说明 |
| --- | --- |
| **速度** | 比标准注意力快 2-4× |
| **显存** | 从 O(N²) 降至 O(N) |
| **精确性** | 数学等价，非近似 |
| **Flash Attention 2** | 进一步优化并行度，最常用 |
| **Flash Attention 3** | 针对 Hopper (H100) 架构优化 |

**PyTorch 2.0+ 原生集成（推荐）**：

```python
# 在支持的硬件上自动使用 Flash Attention，无需额外安装
import torch.nn.functional as F
out = F.scaled_dot_product_attention(query, key, value,
          attn_mask=None, dropout_p=0.0)
```

**手动安装**：`pip install flash-attn --no-build-isolation`（需 CUDA ≥ 11.6，编译耗时较长）

**环境要求**：NVIDIA GPU（Ampere 架构及以上效果最佳），不支持 CPU/MPS。

## 43. Optuna：超参数优化

轻量级超参数优化框架，使用贝叶斯优化（TPE）高效搜索超参空间，自动停止差的试验。

| 工具 | 用途 |
| --- | --- |
| `optuna.create_study(direction)` | 创建优化任务 |
| `study.optimize(fn, n_trials)` | 运行 N 次试验 |
| `trial.suggest_float(name, low, high)` | 建议浮点超参 |
| `trial.suggest_int(name, low, high)` | 建议整数超参 |
| `trial.suggest_categorical(name, choices)` | 建议分类超参 |
| `study.best_params` | 最优超参组合 |
| `optuna.visualization` | 可视化搜索过程 |

安装：`pip install optuna`

In [ ]:
import torch, torch.nn as nn
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        lr    = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
        hdim  = trial.suggest_int("hidden_dim", 8, 64, step=8)
        drop  = trial.suggest_float("dropout", 0.0, 0.5)
        optim = trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"])

        model = nn.Sequential(nn.Linear(4, hdim), nn.ReLU(), nn.Dropout(drop), nn.Linear(hdim, 2))
        opt   = getattr(torch.optim, optim)(model.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()

        torch.manual_seed(0)
        for _ in range(20):
            x, y = torch.randn(32, 4), torch.randint(0, 2, (32,))
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()

        with torch.inference_mode():
            val_loss = loss_fn(model(torch.randn(64, 4)), torch.randint(0, 2, (64,))).item()
        return val_loss

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=20, show_progress_bar=False)
    print(f"完成 {len(study.trials)} 次试验")
    print(f"最优 val_loss: {study.best_value:.4f}")
    print(f"最优超参: {study.best_params}")
except ImportError:
    print("安装: pip install optuna")

## 44. Ray Tune（简介）

基于 Ray 分布式计算引擎的超参数搜索框架，可在多机多 GPU 上并行运行大量试验。

| 工具 | 用途 |
| --- | --- |
| `tune.run(fn, config)` | 启动超参搜索 |
| `tune.grid_search(values)` | 网格搜索 |
| `tune.choice(options)` | 随机选择 |
| `tune.loguniform(low, high)` | 对数均匀分布 |
| `ASHAScheduler` | 早停差的试验（异步连续折半） |
| `PopulationBasedTraining` | 在线超参调整 |

Ray Tune 与 PyTorch Lightning、Transformers Trainer 均有内置集成。

**环境要求**：单机可用，分布式需 Ray 集群。

安装：`pip install "ray[tune]"`

## 45. ONNX：模型导出与跨平台推理

ONNX 是跨框架模型标准格式，导出后可用 ONNX Runtime 在 C++、Java、移动端等环境中推理。

| 工具 | 用途 |
| --- | --- |
| `torch.onnx.export(model, input, path)` | 导出为 `.onnx` 文件 |
| `opset_version` | ONNX 算子集版本（建议 17+） |
| `dynamic_axes` | 声明动态维度（如 batch_size） |
| `onnx.checker.check_model(model)` | 验证模型格式 |
| `onnxruntime.InferenceSession` | 加载并运行 ONNX 模型 |

安装：`pip install onnx onnxruntime`

In [ ]:
import torch, torch.nn as nn, numpy as np

model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
model.eval()
onnx_path = "/tmp/demo.onnx"

torch.onnx.export(
    model, torch.randn(1, 4), onnx_path,
    opset_version=17,
    input_names=["features"], output_names=["logits"],
    dynamic_axes={"features": {0: "batch_size"}, "logits": {0: "batch_size"}},
)
print(f"ONNX 已保存: {onnx_path}")

try:
    import onnx
    m = onnx.load(onnx_path); onnx.checker.check_model(m)
    print("ONNX 验证通过")
    print("输入:", [n.name for n in m.graph.input])
    print("输出:", [n.name for n in m.graph.output])
except ImportError: print("onnx 未安装: pip install onnx")

try:
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    x_np = np.random.randn(4, 4).astype(np.float32)
    ort_out = sess.run(None, {"features": x_np})[0]
    pt_out  = model(torch.tensor(x_np)).detach().numpy()
    print("ONNX Runtime 输出形状:", ort_out.shape)
    print("PyTorch vs ONNX 最大误差:", abs(pt_out - ort_out).max())
except ImportError: print("onnxruntime 未安装: pip install onnxruntime")

## 46. TorchScript：模型序列化与 C++ 部署

将 PyTorch 模型转换为可脱离 Python 运行的静态图，用于 C++ 推理和移动端部署。

| 工具 | 用途 |
| --- | --- |
| `torch.jit.trace(model, input)` | 追踪模式（无控制流） |
| `torch.jit.script(model)` | 脚本模式（支持 if/for/while） |
| `scripted.save(path)` | 保存为 `.pt` 文件 |
| `torch.jit.load(path)` | 加载脚本模型 |
| `@torch.jit.script` | 函数/类装饰器 |

**trace vs script**：trace 不支持 Python 控制流（会被固化），script 支持但要求代码可被 TorchScript 解析。

In [ ]:
import torch, torch.nn as nn

# trace 模式
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
model.eval()
x = torch.randn(2, 4)
traced = torch.jit.trace(model, x)
print("trace 误差:", (model(x) - traced(x)).abs().max().item())
traced.save("/tmp/traced.pt")
loaded = torch.jit.load("/tmp/traced.pt")
print("加载后输出:", loaded(torch.randn(3, 4)).shape)

# script 模式（支持控制流）
class BranchModel(nn.Module):
    def __init__(self): super().__init__(); self.fc = nn.Linear(4, 2)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.fc(x)
        if out.sum() > 0: return torch.relu(out)
        else: return torch.sigmoid(out)

scripted = torch.jit.script(BranchModel())
print("\nscript 输出:", scripted(torch.randn(2, 4)).shape)
print("TorchScript code:\n", scripted.code)

## 47. 其他工具速览

| 工具 | 安装 | 用途 |
| --- | --- | --- |
| **timm** | `pip install timm` | 700+ 预训练视觉模型（ViT、EfficientNet、Swin Transformer 等），比 torchvision 更丰富 |
| **einops** | `pip install einops` | 可读的张量维度操作（`rearrange`、`reduce`、`repeat`），替代复杂的 reshape/permute |
| **bitsandbytes** | `pip install bitsandbytes` | 8-bit/4-bit 量化优化器，QLoRA 的基础依赖 |
| **MLflow** | `pip install mlflow` | 开源实验跟踪 + 模型注册表，可自托管（无需 wandb 账号） |
| **torchinfo** | `pip install torchinfo` | 类似 Keras `model.summary()`，打印每层形状和参数量 |
| **pytest** | `pip install pytest` | 单元测试框架，测试输出形状、数值正确性 |
| **DVC** | `pip install dvc` | 数据版本控制，大文件/数据集的 git |

In [ ]:
import torch, torch.nn as nn

# timm
try:
    import timm
    models = timm.list_models()
    vit    = timm.list_models("vit*")
    print(f"timm 模型总数: {len(models)}, ViT 系列: {len(vit)}")
    m = timm.create_model("efficientnet_b0", pretrained=False, num_classes=10)
    print("EfficientNet-B0:", m(torch.randn(2, 3, 224, 224)).shape)
except ImportError:
    print("timm 未安装: pip install timm")

# einops
try:
    from einops import rearrange, reduce, repeat
    x = torch.randn(8, 3, 32, 32)
    print("\neinops:")
    print("  NCHW→NHWC:", rearrange(x, "b c h w -> b h w c").shape)
    print("  flatten spatial:", rearrange(x, "b c h w -> b c (h w)").shape)
    print("  global avg pool:", reduce(x, "b c h w -> b c", "mean").shape)
    print("  repeat:", repeat(x[0], "c h w -> b c h w", b=4).shape)
except ImportError:
    print("\neinops 未安装: pip install einops")

# torchinfo
try:
    from torchinfo import summary
    model = nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10)
    )
    summary(model, input_size=(4, 3, 32, 32))
except ImportError:
    print("\ntorchinfo 未安装: pip install torchinfo")
    print("用法: from torchinfo import summary; summary(model, input_size=(1,3,224,224))")